# Plotting global mortality timeseries due to PM<sub>2.5</sub> exposure

Using CESM2, SSP2-4.5 ensemble member 1 as an example

In [ ]:
import os
import glob
import numpy as np
import xarray as xr
import matplotlib
import matplotlib.pyplot as plt
from utils.utils import autosize_figure, get_scenario_config
import config
from utils.utils import require_dir
import pathlib

In [ ]:
def process_all_mortality(model, scenario, GBD_version, years, ens_num, n_samples):
    dates = f"{years.start}-{years.stop}"

    # === Load data ===
    DIR = require_dir(pathlib.Path(config.WORK_ROOT) / model / "mortality" / "pm25")
    in_files = f"Global_mortality_{GBD_version}_*_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    in_path = os.path.join(DIR, in_files)
    files = sorted(glob.glob(in_path))

    # Open and combine all mortality outcomes
    datasets = [xr.open_dataarray(f) for f in files]

    # Align (in case of slight coordinate mismatches)
    da = xr.align(*datasets, join="exact")

    return da

In [ ]:
def plotting_data(da, q1, q2):
    mean = da.mean("samples")
    p1 = da.quantile(q1, "samples")
    p2 = da.quantile(q2, "samples")
    return mean, p1, p2

In [ ]:
def plot_stacked_global_mortality(da, q1, q2, years, model, scenario, GBD_version, ens_num, labels):
    dates = f"{years.start}-{years.stop}"
    # Create magma colours for each category
    cmap = matplotlib.colormaps["magma"]
    colors = [cmap(i) for i in np.linspace(0.25, 0.95, len(da))]

    # Extract mean for each health variable (no per-category uncertainty needed)
    means = []
    for item in da:
        da_mean, _, _ = plotting_data(item, q1, q2)
        means.append(da_mean)
    means = np.array(means)

    # Build the cumulative stacking (means only)
    stack_means = np.cumsum(means, axis=0)
    years_vals = da_mean["year"].values

    # Total across categories: sum samples FIRST,
    # then compute mean/quantiles of the summed array.
    da_total = sum(da)
    total_mean, total_p1, total_p2 = plotting_data(da_total, q1, q2)

    plt.figure(figsize=autosize_figure(1, 1, scale_factor=1.2))
    plt.rcParams.update({'font.size': 16})

    # Coloured stacked bands (no individual uncertainty)
    for i in range(len(means)):
        lower_mean = stack_means[i - 1] if i > 0 else 0
        upper_mean = stack_means[i]
        plt.fill_between(
            years_vals,
            lower_mean,
            upper_mean,
            color=colors[i],
            alpha=0.6,
            label=labels[i],
        )

    # Total uncertainty band (hatched, grey)
    plt.fill_between(
        years_vals,
        total_p1,
        total_p2,
        facecolor="lightgrey",
        edgecolor="grey",
        hatch="///",
        linewidth=1,
        alpha=0.4,
    )

    # Total line (black), drawn last so it sits on top
    plt.plot(years_vals, total_mean, color="black", lw=2, label="Total")

    plt.ylim(bottom=0)
    plt.ticklabel_format(style="plain", axis="y")
    plt.grid(True, alpha=0.2)
    plt.ylabel("Total global mortality")
    plt.title(f"{model} {scenario} ensemble {ens_num}")

    # Proxy artist for the uncertainty-band legend entry
    uncertainty_patch = matplotlib.patches.Patch(
        facecolor="lightgrey",
        edgecolor="grey",
        hatch="///",
        alpha=0.4,
        label="95% CI",
    )
    handles, legend_labels = plt.gca().get_legend_handles_labels()
    handles.append(uncertainty_patch)
    legend_labels.append("95% CI")

    plt.legend(
        handles=handles,
        labels=legend_labels,
        frameon=True,
        facecolor="white",
        edgecolor="none",
        framealpha=0.8,
        loc='upper center',
        bbox_to_anchor=(0.5, -0.15),
        ncol=3,   # 9 entries (7 categories + Total + 95% CI) → 3 rows of 3
    )
    ax = plt.gca()
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(axis="x", length=0)
    ax.tick_params(axis="y", length=0)
    plt.tight_layout()
    out_file = f"Global_pm25_mortality_stacked_{GBD_version}_{model}_{scenario}_{dates}.png"
    out_path = os.path.join(SAVE_DIR, out_file)
    plt.savefig(out_path, dpi=300)
    return

In [ ]:
# === Path Config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
SAVE_DIR = require_dir(pathlib.Path(config.PLOTTING_ROOT) / "example_workflow")

model = "CESM2"
scenario = "SSP245"
GBD_version = "GBD23"
ensemble_number = 1

configs = get_scenario_config(model, scenario)
years = configs["years"]

n_samples = 300

# Quantiles for plotting uncertainty
q1 = 0.025
q2 = 0.975
labels = ["COPD", "Dementia", "Type II Diabetes", "Ischemic Heart Disease",
          "Lower Respiratory Infections", "Lung Cancer",
          "Stroke"]

da = process_all_mortality(model, scenario, GBD_version, years, ensemble_number, n_samples)

plot_stacked_global_mortality(da, q1, q2, years, model, scenario, GBD_version, ensemble_number, labels)